In [ ]:
# %pip install PyYAML
# %pip install dynamic-yaml


In [ ]:
# The mandatory items we EXPECT to see
required = {"Revenue per intake", "Revenue per additional learner"}

# Scenario 1: User provided both (Correct)
actual_1 = {"Revenue per intake", "Revenue per additional learner", "Miscellaneous"}
print(required.issubset(actual_1)) 
# Output: True (Because both 'required' items exist in 'actual_1')

# Scenario 2: User provided only one (Error)
actual_2 = {"Revenue per intake", "Deposit Fee"}
print(required.issubset(actual_2)) 
# Output: False (Because 'Revenue per additional learner' is missing)

# Scenario 3: User did not provide any (Error)
actual_3 = set()
print(required.issubset(actual_3))

# Scenarion 4: Just 1 valid in set
actual_4 = {"Revenue per intake"}
print(required.issubset(actual_4))
# Output: False

In [ ]:
import pandas as pd
from loguru import logger
df = pd.DataFrame({"A": [1, 2, 3], "B": [1, 2, 3]}, index=['a', 'a', 'b'])  
logger.info("\n" + df.to_markdown()) 


In [ ]:
import pandas as pd

# Use the calamine engine for the Lhub Finance Migration project data [cite: 20, 33]
xl = pd.ExcelFile('/home/user/datavalidation/data/raw_data/Data Template M01_User Information_C2R2.xlsx', engine="calamine")

# Increase chunksize to at least 10 to see actual data rows 
reader = xl.parse(sheet_name="SalesLocation", chunksize=10, header=0)

for index, chunk in enumerate(reader):
    print(f"--- Processing Chunk {index} ---")
    
    # Check the type to prove it is a DataFrame
    print(f"Object type: {type(chunk)}") 
    
    # Use .head() to see the first few rows clearly
    print(chunk)
    
    # If you need to append to your list for the 1M row migration:
    # chunks.append(chunk)

In [ ]:
import pandas as pd
xl = pd.ExcelFile('/home/user/datavalidation/data/test_data/Data Template M02_Master Data Setup_C2R1.xlsx', engine="calamine")
chunks = []
reader = xl.parse(sheet_name="SuspensionReason", chunksize=3,  dtype=str, header=0)
reader

In [ ]:
import pandas as pd
from loguru import logger
import snoop
import sys
snoop.install(out=sys.stdout, color=True)

@snoop
def read_excel_chunks(file_path, sheet_name, chunksize=10_000):
    # Read the header separately to maintain column consistency
    df_header = pd.read_excel(file_path, sheet_name=sheet_name, nrows=1, engine="openpyxl")
    columns = df_header.columns.tolist()

    skiprows = 1 # Start skipping after the header row
    df = pd.DataFrame([], columns=columns)
    while True:
        # Read a chunk of data, skipping previous rows and without a header
        df_chunk = pd.read_excel(
            file_path,
            sheet_name=sheet_name,
            nrows=chunksize,
            skiprows=skiprows,
            header=None,
            engine="calamine"
        )

        # If the chunk is empty, the end of the file is reached
        if not df_chunk.shape[0]:
            break
        
        # Assign the correct column names
        df_chunk.columns = columns
        
        # Process the chunk (e.g., append to a list, perform analysis)
        # yield df_chunk
        df = pd.concat([df, df_chunk], axis=0, ignore_index=True)

        skiprows += chunksize
    return df

# Example usage:
file_name = '/home/user/datavalidation/data/raw_data/Data Template M01_User Information_C2R2.xlsx'
sheet_name="SalesLocation"
df = read_excel_chunks(file_name, sheet_name)
df


In [ ]:
from dotenv import load_dotenv
import os


load_dotenv()
DATA_FOLDER_PATH = os.getenv("DATA_FOLDER_PATH")
CONSTANTS_CONFIG_FOLDER_PATH = os.getenv("CONSTANTS_CONFIG_FOLDER_PATH")
PROCESSING_CONFIG_FOLDER_PATH = os.getenv("PROCESSING_CONFIG_FOLDER_PATH")
VALIDATION_CONFIG_FOLDER_PATH = os.getenv("VALIDATION_CONFIG_FOLDER_PATH")
DATA_FOLDER_PATH, CONSTANTS_CONFIG_FOLDER_PATH

In [ ]:
import yaml
import pandas as pd

with open("/home/user/datavalidation/configs/file_info.yaml", "r") as f:
    data = yaml.load(f, Loader=yaml.FullLoader)
    df_file_info = pd.DataFrame(data)
    # print(type(data), json.dumps(data, indent=2))
df_file_info = df_file_info.explode("sheets", ignore_index=True)
df_file_info["file"] = DATA_FOLDER_PATH + "/" + df_file_info["file"]
df_file_info["config_file"] = CONSTANTS_CONFIG_FOLDER_PATH + "/" + df_file_info["config_file"]
df_file_info["processing_config_file"] = PROCESSING_CONFIG_FOLDER_PATH + "/" + df_file_info["processing_config_file"]
df_file_info["validation_config_file"] = VALIDATION_CONFIG_FOLDER_PATH + "/" + df_file_info["validation_config_file"]
df_file_info

In [ ]:
common_kwargs = dict()
file_name_mapping = df_file_info[["file_name", "file_name_mapping"]].dropna().drop_duplicates().to_dict(orient="records")
# file_name_mapping = {file_name_mapping.loc[index, "file_name_mapping"]: file_name_mapping.loc[index, "file_name"] for index, item in df_file_info.iterrows()}
file_name_mapping

for item in file_name_mapping:
    common_kwargs[item["file_name_mapping"]] = item["file_name"]

# common_kwargs = {**common_kwargs, **file_name_mapping}
common_kwargs


In [ ]:
pd.read_excel(df_file_info.loc[0, "file"], sheet_name=df_file_info.loc[0, "sheets"])

In [ ]:
import pandas as pd
pd.DataFrame({"A": [True, False, 1, "1"]}).map(lambda x: {
    True: "true",
    False: "false"
}.get(x, x))

In [ ]:
import yaml
import json
import pandas as pd

with open("/home/user/datavalidation/configs/validation/master_data_setup_validation_config.yaml", "r") as f:
    data = yaml.load(f, Loader=yaml.FullLoader)
    df = pd.DataFrame(data)
    # print(type(data), json.dumps(data, indent=2))
df

In [ ]:
df_files = pd.json_normalize(df["files"]).explode("columns", ignore_index=True)
df_columns = pd.json_normalize(df_files["columns"]).explode("rules", ignore_index=True)
df_files = pd.concat([df_files,df_columns], axis=1)
df_rules = pd.json_normalize(df_files["rules"])
df_files = pd.concat([df_files,df_rules], axis=1)

df_files

In [ ]:
df_files["file_path"] = df_files["file_path"].str.replace("${DATA_FOLDER_PATH}", os.environ["DATA_FOLDER_PATH"])
df_files.loc[0, "file_path"]
df_files["report_file_path"] = df_files["report_file_path"].str.replace("${DATA_FOLDER_PATH}", os.environ["DATA_FOLDER_PATH"])
df_files["ref_file"] = df_files["ref_file"].str.replace("${DATA_FOLDER_PATH}", os.environ["DATA_FOLDER_PATH"])

In [ ]:
df_files_copy = df_files.copy()
df_files_copy.drop(["columns", "rules"], axis=1, inplace=True)
df_files_copy[[
    "sheet_name",
    "file_path",
    "file_type",
    "report_file_path"
]] = df_files_copy[[
    "sheet_name",
    "file_path",
    "file_type",
    "report_file_path"
]].ffill()
df_files_copy